# ZelusBench: Sustained Attention — Short

Isolates chain depth as the independent variable.
Controls for confounds: fixed signal-to-noise ratio (num_points = depth * 2),
POSITION-only queries, consistent transform_prob=0.1.

- Depths: [2, 4]
- 10 scenarios per depth = 20 scenarios

In [11]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import time

from zelusbench.scenarios.config import ScenarioConfig, QueryType
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.scorer import score_query, score_response

DEPTHS = [16]
SEEDS = 10
TOTAL = len(DEPTHS) * SEEDS
print(f"ZelusBench Sustained Attention — Short")
print(f"Depths: {DEPTHS} | Seeds: {SEEDS} | Total: {TOTAL} scenarios")

ZelusBench Sustained Attention — Short
Depths: [16] | Seeds: 10 | Total: 10 scenarios


In [ ]:
@kbench.task(name="zelusbench_attn_sustained_short")
def zelusbench_attn_sustained_short(llm) -> tuple[float, float]:
    _start = time.time()
    print(f"Running {TOTAL} scenarios...")
    print("=" * 60)
    all_scores = []
    depth_scores = {}
    scenario_num = 0
    for depth in DEPTHS:
        depth_scores[depth] = []
        for i in range(SEEDS):
            scenario_num += 1
            bg_rng = random.Random(i * 7919)
            cfg = ScenarioConfig.randomize_except(bg_rng, pinned={
                "min_chain_depth": depth, "max_chain_depth": depth,
                "num_points": depth * 2,
                "transform_prob": 0.1,
                "query_types": [QueryType.POSITION],
                "query_min_depth": max(1, depth - 2),
                "num_queries": 3,
                "seed": depth * 1000 + i,
            })
            scenario = ScenarioGenerator(cfg).generate(f"sustained_{depth}_{i}")
            response = llm.prompt(scenario.prompt)
            scores = score_response(response, scenario)
            all_scores.extend(scores)
            avg = float(np.mean([s.score for s in scores]))
            depth_scores[depth].append(avg)
            bg = f"dim={cfg.dim} lb={cfg.leaf_bias} pts={cfg.num_points}"
            print(f"  [{scenario_num}/{TOTAL}] depth={depth} avg={avg:.2f} | {bg}")
        print(f"  >> Depth {depth} mean: {float(np.mean(depth_scores[depth])):.3f}")
    print("\n" + "=" * 60)
    for depth in DEPTHS:
        avg = float(np.mean(depth_scores[depth]))
        print(f"  depth={depth:2d}  accuracy={avg:.3f}")
        kbench.assertions.assert_true(avg >= 0, expectation=f"Depth {depth}: valid (got {avg:.3f})")
    overall = float(np.mean([s.score for s in all_scores]))
    std = float(np.std([s.score for s in all_scores]))
    print(f"\nOverall: {overall:.3f} +/- {std:.3f} | {len(all_scores)} queries | {time.time()-_start:.1f}s")
    return overall, std

zelusbench_attn_sustained_short.run(llm=kbench.llm)

In [15]:
zelusbench_attn_sustained_short()

Running 10 scenarios...
Process the following 3D spatial reasoning scenario. Statements and queries to be processed chronologically.
Format: [Answer q_ID] value — e.g. [Answer q_001] (0.0, 0.0, 0.0) or [Answer q_002] 5.385 or [Answer q_003] B

---

Point A is at offset (-4.0, 3.0, 3.7) from Point O.
Point B is at offset (4.1, -2.1, 1.1) from Point A.
Point C is at offset (1.3, -1.9, -1.7) from Point A.
Point D is the weighted centroid of Point C (weight 0.9), Point A (weight 0.8).
Point E is 1.1 units from Point D at polar angle 94 degrees and azimuthal angle 246 degrees.
Point F is at offset (5.0, 5.4, 0.2) from Point C.
Point G is 2.8 units from Point B at polar angle 27 degrees and azimuthal angle 305 degrees.
Point H is at offset (-1.2, 3.8, 3.7) from Point F.
Point I is 0.8 units from Point D at polar angle 72 degrees and azimuthal angle 19 degrees.
Point J is 1.6 units from Point D at polar angle 76 degrees and azimuthal angle 290 degrees.
Point K is at offset (-5.8, 5.9, 5.8) fr